# Smart microscopy 2: following a path

Two ways to image along a path that is longer than one field of view:

- **A, planned route.** You give waypoints, e.g. along a channel of a microfluidic chip or across
  a well. The notebook samples the route with overlapping fields of view, images it and builds a
  mosaic.
- **B, autonomous following.** The microscope decides where to go next from what it sees. At every
  step it segments the structure, looks ahead in a fan of directions around its current heading,
  and steps towards the best one:
  - `ridge` mode stays on a band of structure, such as a fibre, vessel, root, hypha, channel or
    tissue strip.
  - `edge` mode traces the border between sample and background, for example around a tissue
    section, colony or droplet.

  It stops when the structure ends, the contour closes, or it reaches the step or area limit.

**Safety.** X/Y only, relative steps of at most one field of view, inside a box around the start,
with a step limit. Interrupt the cell (■) to stop.

## 1 Connect

In [ ]:
import json, time, datetime
from pathlib import Path

import numpy as np
import cv2
import matplotlib.pyplot as plt
from IPython.display import display

import imswitchclient.ImSwitchClient as imc
from imswitchclient import smart

# None = $IMSWITCH_API_URL, set by ImSwitch for the notebooks it serves itself.
# From another computer use the browser address, e.g. "http://192.168.178.76/imswitch/api"
IMSWITCH_URL = None

client = imc.ImSwitchClient(IMSWITCH_URL)
print("Connected to:", client.base_uri)
if not hasattr(client, "autofocusManager"):
    raise ImportError("imswitchclient is too old (no smart helpers) - update it, e.g. pip install -U "
                      "https://github.com/openUC2/imswitchclient/archive/refs/heads/main.zip")

STAGE = client.positionersManager.allStageNames[0]
SPEED = 5000                                   # stage speed for all moves
snap = smart.make_snap(client)                 # -> float32 grayscale frame
move = smart.make_relative_mover(client, STAGE, speed=SPEED)   # move(dx_um, dy_um), blocking
WORK_DIR = Path.home() / "smart_microscopy" / "path_following"
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Stage:", STAGE, client.positionersManager.getPositionerPositions()[STAGE])
print("Light sources:", client.lasersManager.getLaserNames())

## 2 Light and a first look

Switch on the illumination you image with. `ILLUMINATION = None` leaves it as it is (e.g. when
you set it up in the ImSwitch UI).

In [ ]:
ILLUMINATION = {"type": "laser", "name": "LED", "value": 1000}
# LED matrix instead:  ILLUMINATION = {"type": "ledmatrix", "intensity": 128}

def light(on=True):
    if ILLUMINATION is None:
        return
    if ILLUMINATION["type"] == "ledmatrix":
        if on:
            client.ledMatrixManager.setAllLED(state=1, intensity=ILLUMINATION["intensity"])
        else:
            client.ledMatrixManager.setAllLEDOff()
    elif on:
        client.lasersManager.setLaserOn(ILLUMINATION["name"], ILLUMINATION["value"])
    else:
        client.lasersManager.setLaserOff(ILLUMINATION["name"])

light(True)
time.sleep(0.5)
frame = snap()
H, W = frame.shape
print(f"frame {W} x {H} px, range {frame.min():.0f}..{frame.max():.0f}")
plt.figure(figsize=(6, 4.5)); plt.imshow(frame, cmap="gray"); plt.axis("off"); plt.show()

## 3 Camera ↔ stage calibration

Closed-loop moves convert "the object is 40 px left of the centre" into a stage move. The notebook
measures that conversion itself. It moves +X and +Y by a few percent of the field of view and finds
the image shift by phase correlation. The measurement includes camera rotation, mirroring and axis
signs, so the loop cannot run the wrong way. The stage returns to where it started.

Use a structured area of the sample. Recalibrate after changing objective, binning or camera. The
result is cached in `WORK_DIR`.

In [ ]:
RECALIBRATE = False
CAL_FILE = WORK_DIR / "stage_camera_calibration.json"


def pixel_size_guess():
    """µm/px from ImSwitch's pixel calibration or objective, 1.0 if neither is known"""
    try:
        r = client.get_json("/PixelCalibrationController/getPixelSizeUm")
        if r.get("pixelSizeUm"):
            return float(r["pixelSizeUm"])
    except Exception:
        pass
    try:
        return float(client.objectiveController.getStatus()["pixelsize"])
    except Exception:
        return 1.0


cal = None
if CAL_FILE.exists() and not RECALIBRATE:
    cal = smart.StageCameraCalibration.load(CAL_FILE)
    if cal.details.get("frame_shape") != [H, W]:
        print("cached calibration was made at another frame size - recalibrating")
        cal = None
if cal is None:
    step_um = 0.15 * min(H, W) * pixel_size_guess()     # ~15 % of the field of view
    print(f"calibrating with {step_um:.1f} µm steps ...")
    cal = smart.StageCameraCalibration.measure(snap, move, step_um=step_um, settle_s=0.3)
    cal.save(CAL_FILE)
print(cal)
print("details:", {k: v for k, v in cal.details.items() if k != "frame_shape"})
FOV_UM = (H * cal.pixel_size_um, W * cal.pixel_size_um)
print(f"field of view ≈ {FOV_UM[1]:.0f} x {FOV_UM[0]:.0f} µm")

## 4 A: Planned route

Waypoints are **relative** to the current position, in µm. Consecutive waypoints are joined by
straight segments that are sampled with `OVERLAP` between neighbouring frames.

In [ ]:
WAYPOINTS_REL_UM = [(0, 0), (600, 0), (600, 400), (0, 400)]
OVERLAP = 0.25
SAVE_TO_IMSWITCH = False      # also write every frame to the ImSwitch data folder


def sample_route(waypoints, spacing):
    points = [np.asarray(waypoints[0], float)]
    for a, b in zip(waypoints[:-1], waypoints[1:]):
        a, b = np.asarray(a, float), np.asarray(b, float)
        n = max(1, int(np.ceil(np.hypot(*(b - a)) / spacing)))
        points += [a + (b - a) * k / n for k in range(1, n + 1)]
    return points


spacing = (1 - OVERLAP) * min(FOV_UM)
route = sample_route(WAYPOINTS_REL_UM, spacing)
length = sum(np.hypot(*(np.subtract(b, a))) for a, b in zip(WAYPOINTS_REL_UM[:-1], WAYPOINTS_REL_UM[1:]))
print(f"{len(route)} frames along {length:.0f} µm ({spacing:.0f} µm apart)")

x0, y0 = smart.get_xy(client, STAGE)
frames_a, positions_a = [], []
here = np.zeros(2)
try:
    for i, p in enumerate(route):
        move(*(p - here))
        here = p
        frames_a.append(snap())
        positions_a.append(smart.get_xy(client, STAGE))
        if SAVE_TO_IMSWITCH:
            client.recordingManager.snapImageToPath(f"route_{i:03d}")
finally:
    move(*(-here))            # back to the start
print("back at", smart.get_xy(client, STAGE))

m = smart.Mosaic(frames_a, positions_a, cal, downsample=2)
fig, ax = plt.subplots(figsize=(9, 7))
m.show(ax)
pts = np.array([m.pixel_of_stage(*p) for p in positions_a])
ax.plot(pts[:, 1], pts[:, 0], "c.-", lw=1)
ax.set_title("planned route")
plt.show()

## 5 B: Autonomous following, settings

Start with the structure you want to follow in the field of view. The first heading is the best
direction over the full circle, unless you set `START_HEADING_DEG`, measured in the image: 0° is
to the right and 90° is down. Tune `MASK` until the overlay in the next cell covers the structure.

In [ ]:
MODE = "ridge"                # "ridge": stay on the structure; "edge": trace its border
SEGMENT = "threshold"         # "threshold": smoothed brightness above a level - broad structures
                              #   (tissue, channels, colonies); "local": smart.foreground_mask -
                              #   thin bright lines on an uneven background
DARK_STRUCTURE = False        # True when the structure is darker than its surroundings
THRESHOLD = None              # threshold mode; None = Otsu on the first frame, then kept fixed
MASK = dict(k=1.0, background_sigma=40.0)   # local mode, see smart.foreground_mask
OCCUPANCY_SIGMA_PX = 8        # smoothing of the mask before deciding
STEP_FRACTION = 0.35          # step length as a fraction of the shorter frame side
N_ANGLES = 48                 # candidate directions over 360°
MAX_TURN_DEG = 70             # per step, relative to the current heading
TURN_PENALTY = 0.15           # prefers going straight on
REVISIT_PENALTY = 0.5         # avoids stepping back onto visited ground
EDGE_SIDE = +1                # edge mode: +1 = sample on the left of the heading, -1 = right
MIN_SCORE = 0.25              # stop when the best direction scores below this
START_HEADING_DEG = None
MAX_STEPS = 30
BOX_HALF_WIDTH_UM = 3000
CLOSE_LOOP = True             # stop when back at the start (edge mode around an object)


def occupancy(frame):
    """0..1 map of 'structure here', smoothed"""
    global THRESHOLD
    if SEGMENT == "threshold":
        g = cv2.GaussianBlur(frame, (0, 0), 4)
        if THRESHOLD is None:
            g8 = cv2.normalize(g, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
            otsu, _ = cv2.threshold(g8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            THRESHOLD = float(g.min() + otsu / 255 * (g.max() - g.min()))
            print(f"THRESHOLD = {THRESHOLD:.1f} (Otsu on this frame)")
        mask = (g < THRESHOLD) if DARK_STRUCTURE else (g > THRESHOLD)
    else:
        mask = smart.foreground_mask(frame, dark_objects=DARK_STRUCTURE, **MASK)
    return cv2.GaussianBlur(mask.astype(np.float32), (0, 0), OCCUPANCY_SIGMA_PX)


def sample_line(occ, angle, r0, r1, n=24, lateral=0.0):
    """Mean occupancy along a ray from the centre, optionally shifted sideways"""
    cy, cx = (occ.shape[0] - 1) / 2, (occ.shape[1] - 1) / 2
    r = np.linspace(r0, r1, n)
    ys = cy + r * np.sin(angle) + lateral * np.cos(angle)
    xs = cx + r * np.cos(angle) - lateral * np.sin(angle)
    ok = (ys >= 0) & (ys <= occ.shape[0] - 1) & (xs >= 0) & (xs <= occ.shape[1] - 1)
    if not ok.any():
        return 0.0
    return float(occ[ys[ok].round().astype(int), xs[ok].round().astype(int)].mean())


STEP_PX = STEP_FRACTION * min(H, W)


def score_directions(occ, heading, visited_px):
    """score per candidate angle (radians) - higher is better"""
    angles = np.linspace(-np.pi, np.pi, N_ANGLES, endpoint=False)
    scores = np.full(N_ANGLES, -np.inf)
    for i, a in enumerate(angles):
        turn = 0.0 if heading is None else np.angle(np.exp(1j * (a - heading)))
        if heading is not None and abs(turn) > np.radians(MAX_TURN_DEG):
            continue
        if MODE == "ridge":
            s = sample_line(occ, a, 0.2 * STEP_PX, STEP_PX)
        else:
            on = sample_line(occ, a, 0.2 * STEP_PX, STEP_PX)
            side = (sample_line(occ, a, 0.2 * STEP_PX, STEP_PX, lateral=-0.3 * STEP_PX)
                    - sample_line(occ, a, 0.2 * STEP_PX, STEP_PX, lateral=+0.3 * STEP_PX))
            s = 1 - 2 * abs(on - 0.5) + 0.5 * EDGE_SIDE * side
        s -= TURN_PENALTY * abs(turn)
        target = np.array([STEP_PX * np.sin(a), STEP_PX * np.cos(a)])
        if any(np.hypot(*(target - v)) < 0.6 * STEP_PX for v in visited_px):
            s -= REVISIT_PENALTY
        scores[i] = s
    return angles, scores


frame = snap()
occ = occupancy(frame)
angles, scores = score_directions(occ, None if START_HEADING_DEG is None else np.radians(START_HEADING_DEG), [])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(frame, cmap="gray"); axes[0].imshow(occ, cmap="magma", alpha=0.4); axes[0].axis("off")
axes[0].set_title("frame + occupancy")
best = angles[np.argmax(scores)]
axes[0].arrow((W - 1) / 2, (H - 1) / 2, STEP_PX * np.cos(best), STEP_PX * np.sin(best),
              color="c", width=2, head_width=10)
axes[1].plot(np.degrees(angles), scores, ".-"); axes[1].axhline(MIN_SCORE, ls="--", c="r")
axes[1].set_xlabel("direction [°] (0 = right, 90 = down)"); axes[1].set_ylabel("score")
fig.tight_layout(); plt.show()
print(f"best start direction {np.degrees(best):.0f}°, score {scores.max():.2f}")

## 6 B: Follow

Each step moves the stage so that the look-ahead point becomes the new centre. Left: the current
frame with the decision. Right: the mosaic so far.

In [ ]:
x0, y0 = smart.get_xy(client, STAGE)
box = smart.SafetyBox(x0, y0, BOX_HALF_WIDTH_UM, max_step_um=0.6 * max(FOV_UM))
heading = None if START_HEADING_DEG is None else np.radians(START_HEADING_DEG)
frames_b, positions_b, headings = [], [], []
handle = display(None, display_id=True)
reason = "step limit"

try:
    for step in range(MAX_STEPS):
        frame = snap()
        sx, sy = smart.get_xy(client, STAGE)
        frames_b.append(frame); positions_b.append((sx, sy))
        # visited stage positions, expressed as image offsets from here
        visited_px = [np.array(cal.content_shift(sx - px, sy - py)) for px, py in positions_b[:-1]]
        occ = occupancy(frame)
        angles, scores = score_directions(occ, heading, visited_px)
        i = int(np.argmax(scores))
        best, best_score = angles[i], scores[i]

        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        axes[0].imshow(frame, cmap="gray"); axes[0].imshow(occ, cmap="magma", alpha=0.35)
        ok = np.isfinite(scores)
        for a, s in zip(angles[ok], scores[ok]):
            axes[0].plot([(W - 1) / 2, (W - 1) / 2 + 0.5 * STEP_PX * np.cos(a)],
                         [(H - 1) / 2, (H - 1) / 2 + 0.5 * STEP_PX * np.sin(a)], c="y", alpha=0.3)
        axes[0].arrow((W - 1) / 2, (H - 1) / 2, STEP_PX * np.cos(best), STEP_PX * np.sin(best),
                      color="c", width=2, head_width=10)
        axes[0].set_title(f"step {step}: {np.degrees(best):.0f}°, score {best_score:.2f}"); axes[0].axis("off")
        mosaic = smart.Mosaic(frames_b, positions_b, cal, downsample=2)
        mosaic.show(axes[1])
        pts = np.array([mosaic.pixel_of_stage(*p) for p in positions_b])
        axes[1].plot(pts[:, 1], pts[:, 0], "c.-")
        fig.tight_layout(); handle.update(fig); plt.close(fig)

        if best_score < MIN_SCORE:
            reason = f"structure lost (score {best_score:.2f})"
            break
        if CLOSE_LOOP and step >= 5 and np.hypot(sx - x0, sy - y0) < 0.75 * STEP_PX * cal.pixel_size_um:
            reason = "back at the start"
            break
        mx, my = cal.move_to_center(STEP_PX * np.sin(best), STEP_PX * np.cos(best))
        mx, my, clipped = box.limit(sx, sy, mx, my)
        if clipped:
            reason = "safety box reached"
            break
        move(mx, my)
        heading = best
        headings.append(best)
except KeyboardInterrupt:
    reason = "interrupted"

path_um = np.sum(np.hypot(*np.diff(np.array(positions_b), axis=0).T)) if len(positions_b) > 1 else 0
print(f"stopped: {reason}. {len(frames_b)} frames along {path_um:.0f} µm of path")

## 7 Result

The full mosaic with the path, saved together with the stage positions. Run the last line to go
back to the start position.

In [ ]:
stamp = f"{datetime.datetime.now():%Y-%m-%d_%H-%M-%S}"
mosaic = smart.Mosaic(frames_b, positions_b, cal)
fig, ax = plt.subplots(figsize=(10, 8))
mosaic.show(ax)
pts = np.array([mosaic.pixel_of_stage(*p) for p in positions_b])
ax.plot(pts[:, 1], pts[:, 0], "c.-"); ax.plot(pts[0, 1], pts[0, 0], "go", ms=10)
ax.set_title(f"{MODE} following: {len(frames_b)} frames ({reason})")
fig.savefig(WORK_DIR / f"path_{stamp}.png", dpi=150, bbox_inches="tight")
plt.show()
np.savez_compressed(WORK_DIR / f"path_{stamp}.npz", frames=np.array(frames_b),
                    positions=np.array(positions_b), A=cal.A)
print("saved to", WORK_DIR)

In [ ]:
# back to where autonomous following started
sx, sy = smart.get_xy(client, STAGE)
move(x0 - sx, y0 - sy)
print("at", smart.get_xy(client, STAGE))